# BAL Score Changes, Teacher Comparison, and Regional Matched Student Counts for Cambridge TKT-Based Professional Development Program Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook follows a step-by-step workflow:
- load dataset metadata from the Croissant schema,
- inspect available record sets and their `@id` values,
- load records into pandas DataFrames,
- perform exploratory data analysis on numeric and categorical fields,
- create simple visualizations for interpretation.

> Important: all dataset entities should be referenced by their Croissant `@id` values whenever available. In this particular package, the provided metadata indicates that the `recordSet` list is empty, so the notebook includes robust checks and a fallback path that still allows useful metadata exploration.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and, if present, records from the dataset using `mlcroissant`.

This section keeps the dataset URL in a variable so the notebook can be easily reused with another Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.f74k-xed8/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

In [ ]:
# High-level metadata summary
summary_rows = [
    ("@id", getattr(metadata, "id", None)),
    ("name", getattr(metadata, "name", None)),
    ("description", getattr(metadata, "description", None)),
    ("version", getattr(metadata, "version", None)),
    ("license", getattr(metadata, "license", None)),
    ("cite_as", getattr(metadata, "cite_as", None)),
]

summary_df = pd.DataFrame(summary_rows, columns=["metadata_field", "value"])
summary_df

## 2. Data Overview
Review available record sets, fields, and their IDs.

The code below inspects the Croissant metadata object without treating it as a dictionary. It attempts to list record sets from the metadata object first. If no record sets are available, the notebook clearly reports that condition.

In [ ]:
# Inspect available record sets from metadata
record_sets = list(getattr(metadata, "record_sets", []) or [])

print(f"Number of record sets discovered: {len(record_sets)}")

record_set_rows = []
for rs in record_sets:
    rs_id = getattr(rs, "id", None)
    rs_name = getattr(rs, "name", None)
    rs_desc = getattr(rs, "description", None)
    num_fields = len(list(getattr(rs, "fields", []) or []))
    record_set_rows.append({
        "record_set_@id": rs_id,
        "record_set_name": rs_name,
        "record_set_description": rs_desc,
        "num_fields": num_fields,
    })

record_sets_df = pd.DataFrame(record_set_rows)
record_sets_df if not record_sets_df.empty else pd.DataFrame([{"message": "No record sets are defined in the dataset metadata."}])

In [ ]:
# List field-level details for each record set using @id references where available
field_rows = []

for rs in record_sets:
    rs_id = getattr(rs, "id", None)
    rs_name = getattr(rs, "name", None)
    for field in list(getattr(rs, "fields", []) or []):
        field_rows.append({
            "record_set_@id": rs_id,
            "record_set_name": rs_name,
            "field_@id": getattr(field, "id", None),
            "field_name": getattr(field, "name", None),
            "field_description": getattr(field, "description", None),
            "field_data_type": getattr(field, "data_type", None),
        })

fields_df = pd.DataFrame(field_rows)
fields_df.head(20) if not fields_df.empty else pd.DataFrame([{"message": "No fields are available because no record sets were defined."}])

The template includes iterating through records from a chosen record set. The next cell follows that pattern, but only executes when a record set `@id` is available.

In [ ]:
# Preview raw records from the first available record set, if any
preview_record_set_id = record_set_rows[0]["record_set_@id"] if record_set_rows else None

if preview_record_set_id is not None:
    print(f"Previewing records from record set @id: {preview_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=preview_record_set_id)):
        print(x)
        if i >= 2:
            break
else:
    print("No record set @id is available, so there are no records to iterate over in this dataset package.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Record set and field references are driven by `@id` values discovered from the metadata.

Because the provided package metadata shows no record sets, this section is written defensively. If record sets are present when the notebook is run, they will be loaded automatically.

In [ ]:
# Extract data from each record set discovered in metadata
record_set_ids = [row["record_set_@id"] for row in record_set_rows if row.get("record_set_@id")]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set @id: {record_set_id}")

if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"\nColumns in first loaded record set ({first_record_set_id}):")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets were loaded because the dataset metadata does not define any record sets.")

When a Croissant package has no accessible `recordSet` definitions, the metadata itself is still valuable for exploration. The next cell converts selected top-level metadata fields into a one-row DataFrame for lightweight analysis.

In [ ]:
# Build a one-row DataFrame from selected metadata attributes as a fallback analytical table
metadata_table = pd.DataFrame([
    {
        "dataset_@id": getattr(metadata, "id", None),
        "name": getattr(metadata, "name", None),
        "description": getattr(metadata, "description", None),
        "version": getattr(metadata, "version", None),
        "license": getattr(metadata, "license", None),
        "date_published": getattr(metadata, "date_published", None),
        "spatial_coverage": getattr(metadata, "spatial_coverage", None),
        "temporal_coverage": getattr(metadata, "temporal_coverage", None),
    }
])

metadata_table

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, grouping, and simple outlier review.

This notebook performs EDA in two layers:
1. **Record-level EDA** if any record set is available.
2. **Metadata-level EDA** as a fallback, using descriptive dataset information and the values stated in the package description.

From the dataset description, the released dataset summarizes:
- **21 teachers**
- **316 matched students**
- variables including teacher code, regional office, matched student count, mean before BAL, mean after BAL, and delta BAL

Since these values are described in metadata but not exposed as a record set in the provided package, we use them carefully as context for summary inspection.

In [ ]:
# Attempt dynamic record-level EDA if any loaded DataFrame contains numeric columns
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs_id, df in dataframes.items():
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        selected_record_set_id = rs_id
        numeric_field_id = numeric_candidates[0]
        non_numeric_candidates = [c for c in df.columns if c != numeric_field_id]
        group_field_id = non_numeric_candidates[0] if non_numeric_candidates else None
        break

if selected_record_set_id is not None:
    analysis_df = dataframes[selected_record_set_id].copy()
    numeric_field = numeric_field_id
    threshold = analysis_df[numeric_field].median()

    filtered_df = analysis_df[analysis_df[numeric_field] > threshold].copy()
    print(f"Filtered records in {selected_record_set_id} with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
    else:
        filtered_df[f"{numeric_field}_normalized"] = 0.0

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field_id} using mean {numeric_field}:")
        display(grouped_df.head())
else:
    print("No record-level numeric fields are available for direct EDA because no tabular record sets were exposed.")

In [ ]:
# Metadata-derived analytical summary using values explicitly stated in the package description
descriptive_context_df = pd.DataFrame([
    {"metric": "teacher_count", "value": 21},
    {"metric": "matched_student_count", "value": 316},
    {"metric": "cells_reported_without_missing_values", "value": 147},
])

descriptive_context_df["normalized_value"] = (
    (descriptive_context_df["value"] - descriptive_context_df["value"].mean())
    / descriptive_context_df["value"].std()
)

threshold = 20
filtered_context_df = descriptive_context_df[descriptive_context_df["value"] > threshold].copy()

print(f"Filtered metadata-derived metrics with value > {threshold}:")
display(filtered_context_df)

print("Normalized metadata-derived metrics:")
display(descriptive_context_df)

In [ ]:
# Parse listed qualitative metadata into simple analysis tables
data_use_cases = list(getattr(metadata, "data_use_cases", []) or [])
data_limitations = list(getattr(metadata, "data_limitations", []) or [])
data_biases = list(getattr(metadata, "data_biases", []) or [])

use_cases_df = pd.DataFrame({"data_use_case": data_use_cases})
limitations_df = pd.DataFrame({"data_limitation": data_limitations})
biases_df = pd.DataFrame({"data_bias": data_biases})

print(f"Number of documented use cases: {len(use_cases_df)}")
print(f"Number of documented limitations: {len(limitations_df)}")
print(f"Number of documented biases: {len(biases_df)}")

display(use_cases_df)
display(limitations_df)
display(biases_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

If record-level data are available, the notebook will plot those directly. Otherwise, it will visualize metadata-derived summaries that are explicitly reported in the dataset description.

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=descriptive_context_df, x="metric", y="value", palette="viridis")
plt.title("Metadata-derived dataset summary metrics")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    x=["use_cases", "limitations", "biases"],
    y=[len(use_cases_df), len(limitations_df), len(biases_df)],
    palette="Set2"
)
plt.title("Counts of documented responsible-use metadata items")
plt.xlabel("Metadata category")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Optional dynamic visualization for record-level data if numeric columns exist
if selected_record_set_id is not None:
    df_plot = dataframes[selected_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df_plot[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No record-level numeric data available for histogram plotting.")

## 6. Conclusion
This exploration shows how to work with a Croissant dataset through `mlcroissant` while keeping references aligned to `@id` values wherever available.

### Key observations
- The dataset metadata loads successfully from the Croissant URL.
- The package contains rich descriptive metadata about scope, limitations, biases, and intended use.
- In the provided package, no `recordSet` entries are exposed, so direct row-level extraction is not available from the schema as supplied.
- Even without record sets, the metadata supports useful documentation review and high-level analysis.

### Recommended next steps
- If an updated Croissant file exposes `recordSet` definitions, rerun this notebook to automatically discover and load those tables.
- Use the documented dataset limitations and bias statements when interpreting any downstream analysis.
- If a tabular distribution becomes available, extend the EDA section to analyze BAL changes, matched student counts, and regional comparisons directly at the released aggregation level.